In [ ]:
import pickle 
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import umap
import matplotlib.pyplot as plt

In [ ]:
#load for radius
#cell_counts around cells 
#fibrosis intensity around cells

#load pixels_outside_cells 
#load cells_subtype_clusters 
#load cluster_color_dict_map

In [ ]:
cluster_type = 'kmeans'

In [ ]:
key = '/Multiplex 6/MP3-058-2'

In [ ]:
cell_clustering_path = 'Y:/coskun-lab/Aditi/01/Results/cell_segmentation_clustering' + key

In [ ]:
cell_pixel_barcode_path = 'Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes' + key

In [ ]:
cell_cluster_location_df = pd.read_csv(cell_clustering_path + '/1x/cell_cluster_location_merged_unclassified_within_contour.csv')

In [ ]:
pixels_outside_cells = pd.read_csv(cell_pixel_barcode_path + '/radius100_repeat_only' + '/pixel_proportions_outside_cells.csv').drop(columns=['Unnamed: 0'])

In [ ]:
with open(cell_pixel_barcode_path + '/radius100_repeat_only' + '/cell_subtype_clusters_' + cluster_type + '.pickle', 'rb') as f: 
    cell_subtype_clusters = pickle.load(f)

In [ ]:
with open(cell_pixel_barcode_path + '/radius100_repeat_only' + '/cluster_color_dict_map_' + cluster_type + '.pickle', 'rb') as f: 
    cluster_color_dict_map = pickle.load(f)

In [ ]:
with open(cell_pixel_barcode_path + '/radius100' + '/cell_counts_outside_cells.pickle', 'rb') as f:
    cell_counts_outside_cells_radius100 = pickle.load(f)

with open(cell_pixel_barcode_path + '/radius100' + '/mean_fl_intensity_per_cell_outside_cells.pickle', 'rb') as f:
    fl_intensity_outside_cells_radius100 = pickle.load(f)

with open(cell_pixel_barcode_path + '/radius100' + '/fibrosis_intensity_counts_outside_cells.pickle', 'rb') as f:
    fibrosis_intensity_counts_outside_cells_radius100 = pickle.load(f)

with open(cell_pixel_barcode_path + '/radius100' + '/weighted_fibrosis_outside_cells.pickle', 'rb') as f:
    fibrosis_outside_cells_radius100 = pickle.load(f)

In [ ]:
cell_counts_outside_cells_radius_df = pd.DataFrame(cell_counts_outside_cells_radius100)
cell_counts_outside_cells_radius_df = cell_counts_outside_cells_radius_df.fillna(0)
cell_proportions_outside_cells_radius_df = cell_counts_outside_cells_radius_df.div(cell_counts_outside_cells_radius_df.sum(axis=1), axis=0)
cell_proportions_outside_cells_radius_df['Cell ID'] = cell_cluster_location_df['Cell ID']
cell_proportions_outside_cells_radius_df['X'] = cell_cluster_location_df['X']
cell_proportions_outside_cells_radius_df['Y'] = cell_cluster_location_df['Y']
cell_proportions_outside_cells_radius_df['Cell Type'] = cell_cluster_location_df['Cell Type']

cell_proportions_outside_cells_radius_df = cell_proportions_outside_cells_radius_df[cell_proportions_outside_cells_radius_df['Cell ID'].isin(pixels_outside_cells['Cell ID'])]


fl_intensity_outside_cells_radius_df = pd.DataFrame(fl_intensity_outside_cells_radius100)
fl_intensity_outside_cells_radius_df['Cell ID'] = cell_cluster_location_df['Cell ID']
fl_intensity_outside_cells_radius_df['X'] = cell_cluster_location_df['X']
fl_intensity_outside_cells_radius_df['Y'] = cell_cluster_location_df['Y']
fl_intensity_outside_cells_radius_df['Cell Type'] = cell_cluster_location_df['Cell Type']

fl_intensity_outside_cells_radius_df = fl_intensity_outside_cells_radius_df[fl_intensity_outside_cells_radius_df['Cell ID'].isin(pixels_outside_cells['Cell ID'])]

fibrosis_intensity_counts_outside_cells_radius_df = pd.DataFrame(fibrosis_intensity_counts_outside_cells_radius100, columns=['Fibrosis Intensity'])

fibrosis_intensity_counts_outside_cells_radius_df['Cell ID'] = cell_cluster_location_df['Cell ID']
fibrosis_intensity_counts_outside_cells_radius_df['X'] = cell_cluster_location_df['X']
fibrosis_intensity_counts_outside_cells_radius_df['Y'] = cell_cluster_location_df['Y']
fibrosis_intensity_counts_outside_cells_radius_df['Cell Type'] = cell_cluster_location_df['Cell Type']

fibrosis_intensity_counts_outside_cells_radius_df = fibrosis_intensity_counts_outside_cells_radius_df[fibrosis_intensity_counts_outside_cells_radius_df['Cell ID'].isin(pixels_outside_cells['Cell ID'])]


fibrosis_outside_cells_radius_df = pd.DataFrame(fibrosis_outside_cells_radius100, columns=['Fibrosis Fraction'])

fibrosis_outside_cells_radius_df['Cell ID'] = cell_cluster_location_df['Cell ID']
fibrosis_outside_cells_radius_df['X'] = cell_cluster_location_df['X']
fibrosis_outside_cells_radius_df['Y'] = cell_cluster_location_df['Y']
fibrosis_outside_cells_radius_df['Cell Type'] = cell_cluster_location_df['Cell Type']

fibrosis_outside_cells_radius_df = fibrosis_outside_cells_radius_df[fibrosis_outside_cells_radius_df['Cell ID'].isin(pixels_outside_cells['Cell ID'])]

cell_environment_radius_df = pd.concat([cell_proportions_outside_cells_radius_df.iloc[:, :5], fibrosis_outside_cells_radius_df.iloc[:, :1], fibrosis_intensity_counts_outside_cells_radius_df], axis=1)   

In [ ]:
cell_environment_radius_df = cell_environment_radius_df.fillna(0)

In [ ]:
cell_types = cell_environment_radius_df['Cell Type'].unique()
for cell_type in cell_types:
    if(cell_type !='Unclassified'):
        cell_environment_radius_df.loc[cell_environment_radius_df['Cell Type'] == cell_type, 'Barcode Cluster'] = cell_subtype_clusters[cell_type]

In [ ]:
import matplotlib as mpl
import seaborn as sns

In [ ]:
subset = 'Cell Type'
subset_key = 'B-cells'
group = 'Barcode Cluster'

In [ ]:
features = ['CD4+ T-cells', 'CD8+ T-cells', 'B-cells', 'Fibrosis Intensity', 'Fibrosis Fraction']

In [ ]:
title_features = ['Neighboring CD4+ T-cells', 'Neighboring CD8+ T-cells', 'Neighboring B-cells', 'Fibrosis Intensity', 'Fibrosis Fraction']

In [ ]:
mpl.rcParams["font.size"] = 20
fig, ax = plt.subplots(1, len(features))
fig.set_figheight(5)
fig.set_figwidth(40)
i = 0
for feature in features:
    colors = list(cluster_color_dict_map[subset_key].values())

    cell_environment_radius_subset = cell_environment_radius_df[cell_environment_radius_df[subset] == subset_key]

    sns.violinplot(
        data=cell_environment_radius_subset,
        x=group,
        y=feature,
        inner='box',  
        cut=0,        
        scale='width', 
        palette=colors, 
        ax=ax[i]
    )

    ax[i].set_xlabel(group)
    ax[i].set_ylabel(title_features[i])
    ax[i].set_xticklabels([str(int(i)+1) for i in list(cluster_color_dict_map[subset_key].keys())])

    plt.tight_layout()
    i+=1 

